In [2]:
import pandas as pd
import numpy as np

In [3]:
# Check 1: Category concentration in ground truth
# For each job, how many L1 categories do its gold skills span?

# Check 2: Category overlap between retrieval and ground truth  
# What % of top-50 retrieved skills share categories with gold skills?

# Check 3: Per-query analysis on the 14 queries where fusion was applied
# Did those 14 queries improve, worsen, or stay same?

In [4]:
# Check 1: Category concentration in ground truth
# For each job, how many L1 categories do its gold skills span?

df_decorte = pd.read_csv(r'../data/title_pairs_desc/decorte_master_3.csv')

In [5]:
df_h = pd.read_csv(
    r'/dss/dsshome1/02/ra95kix2/thesis/skills4cpp/data/processed/master_datasets_2/master_complete_hierarchy_w_occ.csv',
    usecols=['occupationUri', 'skillUri', 'level1_label', 'relationType'],
    dtype=str,
    low_memory=False,
)

In [6]:
df_decorte.merge(df_h, left_on='esco_id', right_on='occupationUri', how='left').groupby('job_id').level1_label.nunique().describe()

count    9885.000000
mean       12.904805
std         2.674418
min         4.000000
25%        11.000000
50%        13.000000
75%        14.000000
max        23.000000
Name: level1_label, dtype: float64

In [7]:
df_decorte.query('split == "test"').merge(df_h, left_on='esco_id', right_on='occupationUri', how='left').groupby('job_id').level1_label.nunique().describe()

count    1046.000000
mean       12.847036
std         2.571750
min         4.000000
25%        11.000000
50%        13.000000
75%        14.000000
max        22.000000
Name: level1_label, dtype: float64

In [9]:
# Check 2: Category overlap between retrieval and ground truth
# What % of top-50 retrieved skills share categories with gold skills?

SIM_PATH = r'/dss/dssmcmlfs01/pr74ze/pr74ze-dss-0001/ra95kix2/outputs/decorte_w_desc_3/similarity_scores.json'
TOPK = 50

# Build: job_id -> set(L1 categories) from ground truth
_gold = (
    df_decorte[['job_id', 'esco_id']]
    .merge(df_h, left_on='esco_id', right_on='occupationUri', how='left')
    .dropna(subset=['job_id', 'level1_label'])
)
job_to_gold_l1 = _gold.groupby(_gold['job_id'].astype(str))['level1_label'].apply(lambda s: set(s.dropna().unique())).to_dict()

# Build: skillUri -> set(L1 categories)
skill_to_l1 = (
    df_h[['skillUri', 'level1_label']]
    .dropna(subset=['skillUri', 'level1_label'])
    .drop_duplicates()
    .groupby('skillUri')['level1_label']
    .apply(lambda s: set(s.dropna().unique()))
    .to_dict()
)

# Stream the huge JSON file (4GB+) without loading it into RAM
try:
    import ijson  # type: ignore
except Exception as e:
    raise ImportError(
        "Missing dependency 'ijson' for streaming JSON parsing. "
        "Install it (e.g. `pip install ijson`) and re-run this cell."
    ) from e

per_job_pct = []
missing_gold_jobs = 0
seen_jobs = 0

total_overlap = 0
total_retrieved = 0

current_job = None
current_item = None
collected_skill_uris = []

with open(SIM_PATH, 'rb') as f:
    for prefix, event, value in ijson.parse(f):
        # top-level keys are job_ids (strings)
        if prefix == '' and event == 'map_key':
            current_job = str(value)
            collected_skill_uris = []
            current_item = None
            continue

        if current_job is None:
            continue

        # Each retrieved result is an object under: <job_id>.item
        if prefix == f'{current_job}.item' and event == 'start_map':
            current_item = {}
            continue

        if current_item is not None and prefix.startswith(f'{current_job}.item.') and event in ('string', 'number'):
            field = prefix.split('.')[-1]
            current_item[field] = value
            continue

        if prefix == f'{current_job}.item' and event == 'end_map':
            if len(collected_skill_uris) < TOPK:
                collected_skill_uris.append(current_item.get('skill_uri'))
            current_item = None
            continue

        # End of the list for this job
        if prefix == current_job and event == 'end_array':
            seen_jobs += 1
            gold_l1 = job_to_gold_l1.get(current_job)
            if not gold_l1:
                missing_gold_jobs += 1
            else:
                overlap = 0
                # treat missing categories as non-overlap (keeps denominator=TOPK)
                for su in collected_skill_uris[:TOPK]:
                    cats = skill_to_l1.get(su)
                    if cats and (cats & gold_l1):
                        overlap += 1

                pct = 100.0 * overlap / TOPK
                per_job_pct.append(pct)
                total_overlap += overlap
                total_retrieved += TOPK

            current_job = None

per_job_pct = np.array(per_job_pct, dtype=float)

print('Jobs seen in similarity_scores:', seen_jobs)
print('Jobs with missing gold categories:', missing_gold_jobs)
print('Overall % of top-50 retrieved skills sharing L1 with gold:', 100.0 * total_overlap / max(total_retrieved, 1))
print('Per-job % stats (only jobs with gold):')
print(pd.Series(per_job_pct).describe(percentiles=[0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95]))

Jobs seen in similarity_scores: 9885
Jobs with missing gold categories: 0
Overall % of top-50 retrieved skills sharing L1 with gold: 96.05300961052099
Per-job % stats (only jobs with gold):
count    9885.000000
mean       96.053010
std         6.139583
min        34.000000
5%         84.000000
10%        90.000000
25%        96.000000
50%        98.000000
75%       100.000000
90%       100.000000
95%       100.000000
max       100.000000
dtype: float64


In [ ]:
# Check 3: Per-query analysis on the ~14 test queries where category weighting was applied

import json

CATEGORY_SCORES_PATH = r'/dss/dssmcmlfs01/pr74ze/pr74ze-dss-0001/ra95kix2/outputs/category_model_h1_soft_deep_larger_val/decorte_w_desc_2_inference/category_scores.json'
FUSED_SCORES_PATH = r'/dss/dssmcmlfs01/pr74ze/pr74ze-dss-0001/ra95kix2/outputs/v2_bayesian_fuser/linear_h1_sum_100/best_fused_scores.json'
BASE_SCORES_PATH = SIM_PATH  # from Check 2

THRESHOLD = 0.40
TOPK_EVAL = 50

# Match the fuser's deduping so job counts align
_df_jobs = df_decorte.copy()
_df_jobs['job_id'] = _df_jobs['job_id'].astype(str)
if 'split' in _df_jobs.columns:
    _df_jobs['split'] = _df_jobs['split'].astype(str).str.lower()
_df_jobs = _df_jobs.drop_duplicates(subset=['raw_title', 'raw_description', 'esco_id', 'job_id'], keep='first')

# Load category scores and compute which jobs pass the threshold (i.e., were eligible for category weighting)
with open(CATEGORY_SCORES_PATH, 'r') as f:
    category_scores = json.load(f)

use_weighting = {}
for job_id, cat_list in category_scores.items():
    # cat_list: [{category, score, ...}, ...]
    max_prob = max((float(x.get('score', 0.0)) for x in cat_list), default=0.0)
    use_weighting[str(job_id)] = (max_prob >= THRESHOLD)

# Identify the "14" test jobs
test_job_ids = set(_df_jobs.loc[_df_jobs['split'] == 'test', 'job_id'].astype(str).tolist())
adjusted_test_job_ids = sorted([jid for jid in test_job_ids if use_weighting.get(jid, False)], key=lambda x: int(x) if x.isdigit() else x)

print('Deduped jobs:', len(_df_jobs))
print('Test jobs:', len(test_job_ids))
print('Adjusted (threshold-pass) test jobs:', len(adjusted_test_job_ids))
print('Adjusted test job_ids:', adjusted_test_job_ids)

# Build gold skills per job_id (using ALL occupation-skill relations from df_h)
occ_to_gold_skills = df_h.dropna(subset=['occupationUri', 'skillUri']).groupby('occupationUri')['skillUri'].apply(lambda s: set(s.astype(str).tolist())).to_dict()
job_to_esco = dict(zip(_df_jobs['job_id'].astype(str), _df_jobs['esco_id'].astype(str)))
job_to_gold_skills = {jid: occ_to_gold_skills.get(job_to_esco.get(jid, ''), set()) for jid in adjusted_test_job_ids}

def ap_at_k(ranked_skill_uris, gold_set, k=50):
    if not gold_set:
        return np.nan
    hits = 0
    sum_prec = 0.0
    for rank, su in enumerate(ranked_skill_uris[:k], start=1):
        if su in gold_set:
            hits += 1
            sum_prec += hits / rank
    return sum_prec / len(gold_set)

def extract_topk_from_base(json_path, job_ids_set, topk=50):
    out = {}
    current_job = None
    current_item = None
    collected = []
    with open(json_path, 'rb') as f:
        for prefix, event, value in ijson.parse(f):
            if prefix == '' and event == 'map_key':
                current_job = str(value)
                collected = []
                current_item = None
                continue
            if current_job is None:
                continue
            if current_job not in job_ids_set:
                # Skip parsing objects for jobs we don't care about
                if prefix == current_job and event == 'end_array':
                    current_job = None
                continue
            if prefix == f'{current_job}.item' and event == 'start_map':
                current_item = {}
                continue
            if current_item is not None and prefix.startswith(f'{current_job}.item.') and event in ('string', 'number'):
                field = prefix.split('.')[-1]
                current_item[field] = value
                continue
            if prefix == f'{current_job}.item' and event == 'end_map':
                if len(collected) < topk:
                    collected.append(current_item.get('skill_uri'))
                current_item = None
                continue
            if prefix == current_job and event == 'end_array':
                out[current_job] = collected[:topk]
                current_job = None
                if len(out) == len(job_ids_set):
                    break
    return out

def extract_topk_from_fused(json_path, job_ids_set, topk=50):
    out = {}
    current_job = None
    current_item = None
    collected = []
    with open(json_path, 'rb') as f:
        for prefix, event, value in ijson.parse(f):
            # job keys are under the "scores" map
            if prefix == 'scores' and event == 'map_key':
                current_job = str(value)
                collected = []
                current_item = None
                continue
            if current_job is None:
                continue
            if current_job not in job_ids_set:
                if prefix == f'scores.{current_job}' and event == 'end_array':
                    current_job = None
                continue
            if prefix == f'scores.{current_job}.item' and event == 'start_map':
                current_item = {}
                continue
            if current_item is not None and prefix.startswith(f'scores.{current_job}.item.') and event in ('string', 'number'):
                field = prefix.split('.')[-1]
                current_item[field] = value
                continue
            if prefix == f'scores.{current_job}.item' and event == 'end_map':
                if len(collected) < topk:
                    collected.append(current_item.get('skill_uri'))
                current_item = None
                continue
            if prefix == f'scores.{current_job}' and event == 'end_array':
                out[current_job] = collected[:topk]
                current_job = None
                if len(out) == len(job_ids_set):
                    break
    return out

# Extract rankings for just those jobs (streaming)
job_ids_set = set(adjusted_test_job_ids)
base_topk = extract_topk_from_base(BASE_SCORES_PATH, job_ids_set, topk=TOPK_EVAL)
fused_topk = extract_topk_from_fused(FUSED_SCORES_PATH, job_ids_set, topk=TOPK_EVAL)

rows = []
for jid in adjusted_test_job_ids:
    gold = job_to_gold_skills.get(jid, set())
    ap_base = ap_at_k(base_topk.get(jid, []), gold, k=TOPK_EVAL)
    ap_fused = ap_at_k(fused_topk.get(jid, []), gold, k=TOPK_EVAL)
    delta = ap_fused - ap_base
    if np.isnan(delta) or abs(delta) < 1e-12:
        outcome = 'same'
    elif delta > 0:
        outcome = 'improve'
    else:
        outcome = 'worsen'

    title = _df_jobs.loc[_df_jobs['job_id'] == jid, 'raw_title'].iloc[0] if (jid in set(_df_jobs['job_id'])) else None

    rows.append({
        'job_id': jid,
        'raw_title': title,
        'ap@50_base': ap_base,
        'ap@50_fused': ap_fused,
        'delta_ap@50': delta,
        'outcome': outcome,
    })

df_cmp = pd.DataFrame(rows).sort_values(['outcome', 'delta_ap@50'], ascending=[True, False])
print('\nOutcome counts:')
print(df_cmp['outcome'].value_counts(dropna=False))

print('\nPer-job details (sorted):')
print(df_cmp[['job_id','raw_title','ap@50_base','ap@50_fused','delta_ap@50','outcome']].to_string(index=False))